# Hypothesis Testing on Model Coefficients

**Objective**: Use statistical hypothesis testing (t-tests) to determine whether each coefficient in our regression model is significantly different from zero. This helps identify features that genuinely contribute to the prediction vs features that may be redundant due to multicollinearity.

**Setup**:
- **Null hypothesis** $H_0$: $a_i = 0$ (the feature has no effect)
- **Alternative hypothesis** $H_1$: $a_i \neq 0$ (the feature contributes to prediction)
- **Test statistic**: $t = \frac{\hat{a}_i}{\text{SE}(\hat{a}_i)}$, where $\text{SE}(\hat{a}_i) = \sqrt{\hat{\sigma}^2 \cdot [(X^TX)^{-1}]_{ii}}$
- **Significance level**: $\alpha = 0.05$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("data/ipl_features.csv")
Y = df['runs_6'].values
n = len(Y)
print(f"Dataset: {n} matches, {int(df['year'].min())}–{int(df['year'].max())}")

In [ ]:
def hypothesis_test(X, Y, feature_names, model_label):
    """
    Perform OLS regression and t-tests on all coefficients.
    
    Returns a DataFrame with coefficients, standard errors, t-stats, 
    p-values, confidence intervals, and significance flags.
    """
    n, p = X.shape
    
    # OLS: a = (X^T X)^(-1) X^T Y
    a = np.linalg.solve(X.T @ X, X.T @ Y)
    
    # Residuals and unbiased variance estimate
    residuals = Y - X @ a
    sigma_sq = np.sum(residuals**2) / (n - p)
    rmse = np.sqrt(np.mean(residuals**2))
    
    # Covariance matrix of coefficients: sigma^2 * (X^T X)^(-1)
    XtX_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(sigma_sq * XtX_inv))
    
    # t-statistics and p-values (two-tailed)
    dof = n - p
    t_stats = a / se
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), dof))
    
    # 95% confidence intervals
    t_crit = stats.t.ppf(0.975, dof)
    ci_lower = a - t_crit * se
    ci_upper = a + t_crit * se
    
    # Significance flags
    sig = []
    for pv in p_values:
        if pv < 0.001: sig.append('***')
        elif pv < 0.01: sig.append('**')
        elif pv < 0.05: sig.append('*')
        else: sig.append('n.s.')
    
    # Build results table
    results = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': a,
        'Std Error': se,
        't-statistic': t_stats,
        'p-value': p_values,
        'CI Lower (95%)': ci_lower,
        'CI Upper (95%)': ci_upper,
        'Significance': sig
    })
    
    print(f"{'=' * 80}")
    print(f"  {model_label}")
    print(f"  RMSE = {rmse:.4f}   |   R² = {1 - np.sum(residuals**2) / np.sum((Y - Y.mean())**2):.4f}   |   DOF = {dof}")
    print(f"{'=' * 80}")
    
    return results, rmse, a

print("hypothesis_test() function defined.")

## Step 1: Test the Full Model (5 features + intercept)

Our original model uses 5 features with an exponential transformation on wickets:

$$Y = a_1 x_1 + a_2 x_2 + a_3 x_3 + a_4 e^{-x_4} + a_5 x_5 + a_0$$

where $x_1$ = runs (excl extras), $x_2$ = dots, $x_3$ = boundaries, $x_4$ = wickets, $x_5$ = extras, and $a_0$ is the intercept.

In [ ]:
x1 = df['runs_excl_extras_3'].values.astype(float)
x2 = df['dots_3'].values.astype(float)
x3 = df['boundaries_3'].values.astype(float)
x4 = np.exp(-df['wickets_3'].values.astype(float))
x5 = df['extras_3'].values.astype(float)

X_full = np.column_stack([x1, x2, x3, x4, x5, np.ones(n)])

results_full, rmse_full, _ = hypothesis_test(
    X_full, Y,
    ['runs_excl_extras (x₁)', 'dots (x₂)', 'boundaries (x₃)', 
     'exp(-wickets) (x₄)', 'extras (x₅)', 'intercept (a₀)'],
    "Model A — Full Model (5 features + intercept)"
)
results_full

### Observation

Two coefficients fail the significance test at $\alpha = 0.05$:

- **exp(-wickets)**: $p = 0.448$ — we cannot reject $H_0$. The 95% confidence interval spans zero.
- **intercept**: $p = 0.210$ — we cannot reject $H_0$.

**Why might this happen?** These features could be collinear with the others. When multiple features are correlated, the variance of each coefficient estimate inflates (the standard error increases), making it harder to establish significance for any single feature. Let's check the cross-correlation matrix to investigate.

## Step 2: Multicollinearity Check

We examine the cross-correlation between all features in the design matrix and compute the **Variance Inflation Factor (VIF)** for each feature. VIF measures how much the variance of a coefficient is inflated due to collinearity with other features. A VIF > 5 indicates problematic multicollinearity.

In [ ]:
# Cross-correlation matrix of the 5 features
feature_df = pd.DataFrame({
    'runs_excl_extras': x1,
    'dots': x2,
    'boundaries': x3,
    'exp(-wickets)': x4,
    'extras': x5,
})

corr_matrix = feature_df.corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(corr_matrix.columns, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)

# Annotate each cell
for i in range(5):
    for j in range(5):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=10, color=color)

plt.colorbar(im, ax=ax, label='Pearson Correlation')
plt.title('Feature Cross-Correlation Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("plots/correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Variance Inflation Factor (VIF) for each feature
# VIF_j = 1 / (1 - R²_j), where R²_j is from regressing feature j on all other features

features = [x1, x2, x3, x4, x5]
feat_names = ['runs_excl_extras', 'dots', 'boundaries', 'exp(-wickets)', 'extras']

print("Variance Inflation Factors (VIF)")
print("=" * 45)
print(f"{'Feature':20s} {'VIF':>8s}  {'Status':>12s}")
print("-" * 45)

for j in range(5):
    others = np.column_stack([features[i] for i in range(5) if i != j] + [np.ones(n)])
    target = features[j]
    a = np.linalg.solve(others.T @ others, others.T @ target)
    pred = others @ a
    ss_res = np.sum((target - pred)**2)
    ss_tot = np.sum((target - target.mean())**2)
    r_sq = 1 - ss_res / ss_tot
    vif = 1 / (1 - r_sq)
    status = "HIGH" if vif > 5 else "OK"
    print(f"{feat_names[j]:20s} {vif:8.2f}  {status:>12s}")

print("-" * 45)
print("\nVIF > 5 indicates problematic multicollinearity.")
print("VIF > 10 indicates severe multicollinearity.")

### Findings

The correlation matrix reveals strong multicollinearity:
- **runs_excl_extras & boundaries**: $r = 0.924$ — extremely high. This makes sense: more boundaries = more runs.
- **runs_excl_extras & dots**: $r = -0.636$ — strong negative correlation. More dots = fewer runs.
- **exp(-wickets)** correlates moderately with dots ($r = 0.329$) and runs ($r = -0.283$).

The VIF analysis quantifies the damage: runs_excl_extras and boundaries have high VIFs because they're nearly linear combinations of each other. This inflates the standard errors of their coefficients.

The exp(-wickets) term, while physically meaningful, is partially captured by the other features — when wickets fall, dots tend to increase and runs tend to decrease. This shared information makes its coefficient estimate noisy, hence the high p-value.

The intercept is also non-significant because when all features are zero (no runs, no dots, no boundaries, $e^0 = 1$, no extras), the model's prediction is anchored by the other terms. The intercept is absorbing whatever the other features don't explain at the origin, but the data doesn't strongly constrain its value.

## Step 3: Remove Non-Significant Terms

Since exp(-wickets) and the intercept both fail the significance test, we remove them and re-fit the model:

$$Y = a_1 x_1 + a_2 x_2 + a_3 x_3 + a_5 x_5$$

We then perform the hypothesis test again on the reduced model to verify all remaining coefficients are significant.

In [ ]:
X_reduced = np.column_stack([x1, x2, x3, x5])

results_reduced, rmse_reduced, a_reduced = hypothesis_test(
    X_reduced, Y,
    ['runs_excl_extras (x₁)', 'dots (x₂)', 'boundaries (x₃)', 'extras (x₅)'],
    "Model D — Reduced Model (4 features, no intercept)"
)
results_reduced

### Result

After removing exp(-wickets) and the intercept, **all 4 remaining coefficients are highly significant** with $p < 0.00001$. The t-statistics are much larger in magnitude and the confidence intervals are tighter, indicating more precise coefficient estimates.

The RMSE barely changed (from 9.075 to 9.095), confirming that the removed terms were contributing very little predictive signal.

## Step 4: Visualize the Comparison

Let's compare the p-values across both models and visualize the confidence intervals to see the improvement clearly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: p-value comparison (bar chart) ---
ax = axes[0]

# Full model p-values
full_names = ['runs', 'dots', 'bounds', 'e⁻ʷᵏᵗ', 'extras', 'intercept']
full_pvals = results_full['p-value'].values
# Reduced model p-values (pad with NaN for missing features)
red_names = ['runs', 'dots', 'bounds', 'extras']
red_pvals = results_reduced['p-value'].values

x_pos = np.arange(len(full_names))
bar_width = 0.35

bars1 = ax.bar(x_pos - bar_width/2, -np.log10(np.clip(full_pvals, 1e-20, 1)), 
               bar_width, label='Model A (full)', color='#e74c3c', alpha=0.8)

# Map reduced model features to full model positions
red_positions = [0, 1, 2, 4]  # runs, dots, bounds, extras
red_bar_vals = np.zeros(len(full_names))
for i, pos in enumerate(red_positions):
    red_bar_vals[pos] = -np.log10(max(red_pvals[i], 1e-20))

bars2 = ax.bar(x_pos + bar_width/2, red_bar_vals, 
               bar_width, label='Model D (reduced)', color='#2ecc71', alpha=0.8)

ax.axhline(y=-np.log10(0.05), color='gray', linestyle='--', linewidth=1, label='α = 0.05')
ax.axhline(y=-np.log10(0.001), color='gray', linestyle=':', linewidth=1, label='α = 0.001')
ax.set_xticks(x_pos)
ax.set_xticklabels(full_names, fontsize=9)
ax.set_ylabel('-log₁₀(p-value)', fontsize=11)
ax.set_title('Statistical Significance of Coefficients', fontsize=12)
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(0, max(-np.log10(np.clip(full_pvals, 1e-20, 1)).max(), 
                     -np.log10(np.clip(red_pvals, 1e-20, 1)).max()) * 1.15)

# Annotate the non-significant ones
for i in [3, 5]:  # exp(-wickets) and intercept
    if full_pvals[i] > 0.05:
        ax.annotate('n.s.', xy=(x_pos[i] - bar_width/2, -np.log10(full_pvals[i])), 
                    ha='center', va='bottom', fontsize=8, color='red', fontweight='bold')

# --- Right: Confidence interval comparison ---
ax = axes[1]

# Full model CIs
full_coefs = results_full['Coefficient'].values
full_ci_low = results_full['CI Lower (95%)'].values
full_ci_high = results_full['CI Upper (95%)'].values

y_pos_full = np.arange(len(full_names))
ax.errorbar(full_coefs, y_pos_full + 0.15, 
            xerr=[full_coefs - full_ci_low, full_ci_high - full_coefs],
            fmt='o', color='#e74c3c', capsize=4, markersize=6, label='Model A (full)')

# Reduced model CIs
red_coefs = results_reduced['Coefficient'].values
red_ci_low = results_reduced['CI Lower (95%)'].values
red_ci_high = results_reduced['CI Upper (95%)'].values

red_y = [0, 1, 2, 4]  # map to same y positions
ax.errorbar(red_coefs, [y - 0.15 for y in red_y],
            xerr=[red_coefs - red_ci_low, red_ci_high - red_coefs],
            fmt='s', color='#2ecc71', capsize=4, markersize=6, label='Model D (reduced)')

ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.set_yticks(y_pos_full)
ax.set_yticklabels(full_names, fontsize=9)
ax.set_xlabel('Coefficient Value', fontsize=11)
ax.set_title('95% Confidence Intervals', fontsize=12)
ax.legend(fontsize=8)
ax.invert_yaxis()

# Highlight CIs that cross zero
for i in range(len(full_names)):
    if full_ci_low[i] < 0 < full_ci_high[i]:
        ax.axhspan(i - 0.4, i + 0.4, color='red', alpha=0.05)

plt.tight_layout()
plt.savefig("plots/hypothesis_testing.png", dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Cross-Validate Both Models

Statistical significance tells us which coefficients are well-estimated, but it doesn't directly tell us which model **predicts** better. We compare Models A and D on held-out test data.

In [ ]:
np.random.seed(42)
n_splits = 200

results_cv = {'Model A (full, 6 params)': [], 'Model D (reduced, 4 params)': []}
matrices = {'Model A (full, 6 params)': X_full, 'Model D (reduced, 4 params)': X_reduced}

for _ in range(n_splits):
    idx = np.random.permutation(n)
    split = int(0.8 * n)
    tr, te = idx[:split], idx[split:]
    
    for name, X in matrices.items():
        a = np.linalg.solve(X[tr].T @ X[tr], X[tr].T @ Y[tr])
        results_cv[name].append(np.sqrt(np.mean((X[te] @ a - Y[te])**2)))

# Plot distribution of test RMSEs
fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(7, 12, 40)
ax.hist(results_cv['Model A (full, 6 params)'], bins=bins, alpha=0.6, 
        color='#e74c3c', label=f'Model A: {np.mean(results_cv["Model A (full, 6 params)"]):.3f} ± {np.std(results_cv["Model A (full, 6 params)"]):.3f}')
ax.hist(results_cv['Model D (reduced, 4 params)'], bins=bins, alpha=0.6, 
        color='#2ecc71', label=f'Model D: {np.mean(results_cv["Model D (reduced, 4 params)"]):.3f} ± {np.std(results_cv["Model D (reduced, 4 params)"]):.3f}')
ax.axvline(np.mean(results_cv['Model A (full, 6 params)']), color='#e74c3c', linestyle='--', linewidth=1.5)
ax.axvline(np.mean(results_cv['Model D (reduced, 4 params)']), color='#2ecc71', linestyle='--', linewidth=1.5)
ax.set_xlabel('Test RMSE', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title(f'Test RMSE Distribution ({n_splits} Random 80-20 Splits)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("plots/cv_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nModel A (full):    mean test RMSE = {np.mean(results_cv['Model A (full, 6 params)']):.4f} ± {np.std(results_cv['Model A (full, 6 params)']):.4f}")
print(f"Model D (reduced): mean test RMSE = {np.mean(results_cv['Model D (reduced, 4 params)']):.4f} ± {np.std(results_cv['Model D (reduced, 4 params)']):.4f}")
print(f"Difference: {np.mean(results_cv['Model D (reduced, 4 params)']) - np.mean(results_cv['Model A (full, 6 params)']):+.4f}")

## Step 6: Summary

| | Model A (full) | Model D (reduced) |
|---|---|---|
| **Features** | runs, dots, boundaries, exp(-wickets), extras + intercept | runs, dots, boundaries, extras |
| **Parameters** | 6 | 4 |
| **In-sample RMSE** | 9.075 | 9.095 |
| **Test RMSE (CV)** | ~9.20 | ~9.24 |
| **All coefficients significant?** | No (exp(-wickets) p=0.45, intercept p=0.21) | Yes (all p < 0.00005) |

### Conclusion

The hypothesis testing procedure reveals that:

1. **In the full model**, exp(-wickets) and the intercept are not statistically significant at $\alpha = 0.05$, likely due to multicollinearity — their effects are partially captured by the other features.

2. **Removing these terms** yields a cleaner model where every coefficient has a p-value close to zero, meaning each feature's contribution is statistically well-supported.

3. **However**, Model A still has a marginally better test RMSE (~0.05 lower). This is because the t-test evaluates whether we can confidently estimate a coefficient's value, not whether the feature helps prediction. A coefficient can be genuinely useful even if its estimate is noisy — across many test samples, the small corrections from exp(-wickets) and the intercept add up.

4. **For the submission**, we use Model A (the full model) since the evaluation metric is RMSE and every fraction of a run counts. For statistical reporting and interpretation, Model D is the more defensible choice since all coefficients are rigorously justified.

This is a classic **bias-variance tradeoff**: Model D has lower variance (more stable estimates) but slightly higher bias (missing the small wicket effect), while Model A has higher variance but lower bias.